In [ ]:
# This code runs several evolutionary strategies with different parameters
# for N_RUNS, computes AUC from the fitness curve, and performs 
# Wilcoxon rank sum test to evaluate significant differences between the strategies
# (tests if the difference between the curves are systematic and not random)
from src.controller.ga import GeneticAlgorithm, GAConfig
from src.model.molecule import Molecule
from src.model.population import Population
from src.model.fitness import compute_fitness, compute_fitness_penalized, novelty_augmented_fitness, compute_dummy_fitness
from src.model.fitness import archive
from src.model.stats import mean_fitness_curve, auc_from_curve, wilcoxon_rank_sum, cliffs_delta
# quiet runner + stats pipeline (prints only run/strategy)
from contextlib import redirect_stdout, redirect_stderr

# initial digital soup
soup = ['[C][#N]', '[C][=O]', '[C][O]', '[C][C][O]', '[C][C][=O]', '[O][=C][C][O]', '[O][=C][O]', '[N][C][=Branch1][C][=O][N]', '[N]', '[O]', '[N][C][C][=Branch1][C][=O][O]', '[C][C][Branch1][=Branch1][C][=Branch1][C][=O][O][N]', '[C][C][=Branch1][C][=O][O]', '[C][C][N]', '[C][S]', '[C][C][=Branch1][C][=O][C][=Branch1][C][=O][O]', '[C][C][=Branch1][C][=O][C]', '[O][=C][=O]', '[O][=C][=S]', '[O][P][=Branch1][C][=O][Branch1][C][O][O]', '[C][=C][C][=C][C][=C][Ring1][=Branch1]', '[C][=C][N][=C][NH1][Ring1][Branch1]', '[C][C][=C][NH1][C][=Ring1][Branch1]', '[C][C][C][C][C][Ring1][Branch1]', '[C][C][C][C][C][C][Ring1][=Branch1]', '[N][C][=N][C][=C][N][Ring1][Branch1]', '[C][C][=C][O][C][=Ring1][Branch1]', '[O][C][C][=Branch1][C][=O][O]', '[C][=N][C][=N][C][NH1][C][=N][C][Ring1][=Branch2][=Ring1][Branch1]', '[O][P][=Branch1][C][=O][Branch1][C][O][O][P][=Branch1][C][=O][Branch1][C][O][O]', '[C][C][Branch1][C][O][C][=Branch1][C][=O][O]', '[O][=C][C][Branch1][C][O][C][O]', '[O][=C][Branch1][Ring1][C][O][C][O]', '[N][C][=O]', '[C][=C]', '[C][C][C][=Branch1][C][=O][O]', '[O][=C][Branch1][C][O][C][C][C][=Branch1][C][=O][O]', '[N][C][C][S]', '[N][C][=Branch1][C][=S][N]', '[O][C][C@H1][O][C][Branch1][C][O][C@H1][Branch1][C][O][C@@H1][Ring1][#Branch1][O]']


class _DevNull:
    def write(self, *args, **kwargs): pass
    def flush(self): pass

def run_quiet_evolve(ga, pop, generations):
    """Run GA"""
    with redirect_stdout(_DevNull()), redirect_stderr(_DevNull()):
        return ga.evolve(pop, generations=generations)

N_RUNS = 20
GENERATIONS = 25
SEEDS = list(range(N_RUNS))

# auc lists for currents strategies used
aucs_penalized = []
aucs_novelty_01 = []
aucs_novelty_05 = []

for seed in SEEDS:
    # strategy 1: penalized fitness (without novelty component)
    print(f"[Run {seed:02d}] penalized")
    archive.archive.clear()
    pop = Population([Molecule(s) for s in soup])

    # selection and replacement parameters
    cfg = GAConfig(
        mu=50, lam=50,
        mutation_rate=0.4, crossover_rate=0.4,
        tournament_k=2, rank_bias=1.7,
        random_seed=seed
    )
    # run the GA with the current strategy
    ga = GeneticAlgorithm(cfg, compute_fitness_penalized)
    history = run_quiet_evolve(ga, pop, GENERATIONS)

    # compute curve and AUC
    curve = mean_fitness_curve(history)
    aucs_penalized.append(auc_from_curve(curve))

    # strategy 2: novelty weight = 0.1
    print(f"[Run {seed:02d}] novelty 0.1")
    archive.archive.clear()
    pop = Population([Molecule(s) for s in soup])

    # selection and replacement parameters
    cfg = GAConfig(
        mu=50, lam=50,
        mutation_rate=0.4, crossover_rate=0.4,
        tournament_k=2, rank_bias=1.7,
        random_seed=seed
    )
    # run the GA with the current strategy
    ga = GeneticAlgorithm(cfg, lambda m: novelty_augmented_fitness(m, novelty_weight=0.1))
    history = run_quiet_evolve(ga, pop, GENERATIONS)

    # compute curve and AUC
    curve = mean_fitness_curve(history)
    aucs_novelty_01.append(auc_from_curve(curve))

    # strategy 3: novelty weight = 0.5
    print(f"[Run {seed:02d}] novelty 0.5")
    archive.archive.clear()
    pop = Population([Molecule(s) for s in soup])

    # selection and replacement parameters
    cfg = GAConfig(
        mu=50, lam=50,
        mutation_rate=0.4, crossover_rate=0.4,
        tournament_k=2, rank_bias=1.7,
        random_seed=seed
    )
    # # run the GA with the current strategy
    ga = GeneticAlgorithm(cfg, lambda m: novelty_augmented_fitness(m, novelty_weight=0.5))
    history = run_quiet_evolve(ga, pop, GENERATIONS)

    # compute curve and AUC
    curve = mean_fitness_curve(history)
    aucs_novelty_05.append(auc_from_curve(curve))

print("\nDone collecting AUCs.")
print("AUCs (penalized):", aucs_penalized)
print("AUCs (novelty 0.1):", aucs_novelty_01)
print("AUCs (novelty 0.5):", aucs_novelty_05)

# Wilcoxon rank-sum test
p_p_vs_n01 = wilcoxon_rank_sum(aucs_penalized, aucs_novelty_01)
d_p_vs_n01 = cliffs_delta(aucs_penalized, aucs_novelty_01)

p_p_vs_n05 = wilcoxon_rank_sum(aucs_penalized, aucs_novelty_05)
d_p_vs_n05 = cliffs_delta(aucs_penalized, aucs_novelty_05)

p_n01_vs_n05 = wilcoxon_rank_sum(aucs_novelty_01, aucs_novelty_05)
d_n01_vs_n05 = cliffs_delta(aucs_novelty_01, aucs_novelty_05)

print("\n--- Wilcoxon rank-sum p-values + Cliff's delta ---")
print(f"penalized vs novelty(0.1): p={p_p_vs_n01:.4g}, delta={d_p_vs_n01:.3f}")
print(f"penalized vs novelty(0.5): p={p_p_vs_n05:.4g}, delta={d_p_vs_n05:.3f}")
print(f"novelty(0.1) vs novelty(0.5): p={p_n01_vs_n05:.4g}, delta={d_n01_vs_n05:.3f}")

